# VirtualFit fashion SLM: cloud LoRA training

## Goal
Fine-tune the language-only Qwen2.5-0.5B-Instruct model with LoRA on a free Colab CUDA runtime. This notebook has **not been executed on a GPU**. It makes no vision-model or training-success claims.

Inputs: this project's source-only training ZIP and 72 independently authored synthetic train / 11 validation / 17 test examples, including Nova identity examples, separate from the 102-case held-out API suite. This tiny paraphrase corpus is a pipeline starter, not enough for a production fashion assistant. No private wardrobe data is uploaded. Base license: Apache-2.0. GPU availability and Colab quotas are outside the project's control.

## Setup
Select Runtime ? Change runtime type ? T4 GPU before running. The guard below stops before model downloads on CPU.

In [ ]:
import subprocess, sys
subprocess.run(['nvidia-smi'], check=True)
# Run model work in fresh subprocesses so installs never leave stale imported libraries.
subprocess.check_call([sys.executable, '-c', "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime'; print({'torch':torch.__version__, 'cuda':torch.version.cuda, 'gpu':torch.cuda.get_device_name(0), 'free_total_bytes':torch.cuda.mem_get_info()})"])


### Upload the source-only bundle
On your computer run `python slm-engine/tools/package_training.py`, then upload `slm-engine/runs/virtualfit-slm-training.zip`. The packer excludes weights, runtime binaries, uploads and secrets.

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files
uploaded = files.upload()
archive_path = Path("virtualfit-slm-training.zip")
assert archive_path.name in uploaded
destination = Path("/content").resolve()
with zipfile.ZipFile(archive_path) as archive:
    assert all((destination / entry.filename).resolve().is_relative_to(destination) for entry in archive.infolist())
    archive.extractall(destination)
slm_root = destination / "slm-engine"
assert (slm_root / "pyproject.toml").is_file()

### Install language-training dependencies
PyTorch is supplied by Colab. Plain LoRA is the default for this small model. QLoRA is optional and requires a compatible Linux/CUDA bitsandbytes installation; GGUF is only for inference, never a training checkpoint.

In [ ]:
import subprocess, sys, json
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(slm_root / 'requirements-colab.txt')])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(slm_root)])
subprocess.check_call([sys.executable, '-m', 'pip', 'check'])
USE_QLORA = False  # Plain fp16 LoRA: batch 1, length 1024, rank 8, accumulation 8.
# First run uses plain LoRA; use a NEW RUN identity if experimenting with QLoRA.
print(json.loads((slm_root / 'configs/lora.json').read_text()))


## Steps
### Validate data and train
All splits are validated for request overlap and wardrobe-ID integrity. Only assistant tokens contribute to loss. The script requires CUDA, checks finite losses and completed optimizer steps, and saves a provenance manifest. Do not infer training success until this cell completes and the manifest is present.

In [ ]:
preflight = r"""
import json
from pathlib import Path
from transformers import AutoTokenizer, AutoConfig
from virtualfit_slm.data import load_records, validate_splits
from virtualfit_slm.train import encode_record
root = Path('/content/slm-engine')
cfg = json.loads((root/'configs/lora.json').read_text())
tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'], revision=cfg['revision'], trust_remote_code=False)
model_config = AutoConfig.from_pretrained(cfg['base_model'], revision=cfg['revision'], trust_remote_code=False)
assert model_config.model_type == 'qwen2' and tokenizer.chat_template
lengths = [len(encode_record(tokenizer, row, cfg['max_length'])['input_ids']) for split in validate_splits(root/cfg['data_dir']) for row in load_records(root/cfg['data_dir']/f'{split}.jsonl')]
print({'model_type': model_config.model_type, 'resolved_revision':model_config._commit_hash, 'examples':len(lengths), 'max_tokens':max(lengths)})
assert model_config._commit_hash == cfg['revision']
"""
subprocess.check_call([sys.executable, '-c', preflight])


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
sys.path.insert(0, str(slm_root))
from virtualfit_slm.data import validate_splits
DATA = slm_root / 'data/training-v2'
print(validate_splits(DATA))
RUN = Path('/content/drive/MyDrive/virtualfit-slm/experiment-v2')
RUN.mkdir(parents=True, exist_ok=True)
base_command = [sys.executable, '-m', 'virtualfit_slm.train', '--config', str(slm_root / 'configs/lora.json'), '--data-dir', str(DATA)]
if USE_QLORA: base_command.append('--qlora')
smoke = RUN / 'smoke'
if not (smoke / 'training-manifest.json').exists():
    smoke_checkpoints = sorted(smoke.glob('checkpoint-*'))
    command = base_command + ['--max-steps', '2', '--output-dir', str(smoke)]
    if smoke_checkpoints: command += ['--resume', str(smoke_checkpoints[-1])]
    subprocess.check_call(command)
smoke_manifest = json.loads((smoke / 'training-manifest.json').read_text())
assert smoke_manifest['optimizer_steps_completed'] == 2
assert (smoke / 'adapter_model.safetensors').stat().st_size > 0
checkpoint = smoke / 'checkpoint-000002'
assert (checkpoint / 'trainer-state.pt').is_file()
print({'smoke_steps': 2, 'validation_loss': smoke_manifest['validation_loss']})
import hashlib
expected_hashes = {split: hashlib.sha256((DATA / f'{split}.jsonl').read_bytes()).hexdigest() for split in ['train', 'validation', 'test']}
assert smoke_manifest['dataset_sha256'] == expected_hashes, 'Use a new RUN directory for changed data'
assert smoke_manifest['resolved_revision'] == json.loads((slm_root/'configs/lora.json').read_text())['revision']
assert smoke_manifest['changed_adapter_parameters'], 'No real weight updates'


In [ ]:
subprocess.check_call([sys.executable, '-m', 'virtualfit_slm.compare_checkpoints', '--adapter', str(smoke), '--data', str(DATA), '--output', str(RUN / 'smoke-reload-comparison.json')])


### Resume the smoke checkpoint into the full run
Optimizer, scaler, Python/PyTorch/CUDA RNG, shuffled order and completed steps are restored. Model revision, dataset hashes and optimization settings must match. Checkpoints are saved every four steps and at the final step. After a Colab disconnect, rerun setup and this cell: it chooses the latest persistent checkpoint. Use a new RUN directory for incompatible experiments.

In [ ]:
adapter = RUN / 'fashion-lora'
checkpoints = sorted(adapter.glob('checkpoint-*')) if adapter.exists() else []
resume = checkpoints[-1] if checkpoints else checkpoint
manifest_file = adapter / 'training-manifest.json'
if not manifest_file.exists():
    subprocess.check_call(base_command + ['--max-steps', '64', '--output-dir', str(adapter), '--resume', str(resume)])
manifest = json.loads(manifest_file.read_text())
assert manifest['optimizer_steps_completed'] == 64
assert manifest['base_model'] == 'Qwen/Qwen2.5-0.5B-Instruct'
assert (adapter / 'adapter_model.safetensors').stat().st_size > 0
print({key: manifest[key] for key in ['optimizer_steps_completed', 'validation_loss', 'dataset_counts', 'resolved_revision']})
assert manifest['dataset_sha256'] == expected_hashes
assert manifest['resolved_revision'] == json.loads((slm_root/'configs/lora.json').read_text())['revision']

### Merge the trained adapter
Merging materializes the full model; run on Colab rather than the memory-constrained development computer.

In [ ]:
subprocess.check_call([sys.executable, '-m', 'virtualfit_slm.compare_checkpoints', '--adapter', str(adapter), '--data', str(DATA), '--output', str(RUN / 'heldout-base-vs-lora.json')])


In [ ]:
merged = slm_root / "runs/fashion-merged"
subprocess.check_call([sys.executable, "-m", "virtualfit_slm.export", str(adapter), str(merged)])

### Export GGUF and quantize for CPU inference
Uses the same pinned llama.cpp revision as local inference. Building the CPU quantizer in Colab may take several minutes. Conversion dependencies are installed from the pinned source tree.

In [ ]:
llama_source = Path("/content/llama.cpp")
if not llama_source.exists(): subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "b11643", "https://github.com/ggml-org/llama.cpp.git", str(llama_source)])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(llama_source / "requirements/requirements-convert_hf_to_gguf.txt")])
assert subprocess.check_output(["git", "-C", str(llama_source), "describe", "--tags", "--exact-match"], text=True).strip() == "b11643"
fp16 = RUN / "virtualfit-fashion-f16.gguf"
quantized = RUN / "virtualfit-fashion-q4_k_m.gguf"
subprocess.check_call([sys.executable, str(llama_source / "convert_hf_to_gguf.py"), str(merged), "--outfile", str(fp16), "--outtype", "f16"])
subprocess.check_call(["cmake", "-S", str(llama_source), "-B", str(llama_source / "build"), "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF", "-DLLAMA_BUILD_TESTS=OFF"])
subprocess.check_call(["cmake", "--build", str(llama_source / "build"), "--target", "llama-quantize", "-j", "2"])
subprocess.check_call([str(llama_source / "build/bin/llama-quantize"), str(fp16), str(quantized), "Q4_K_M"])

## Checks
Keep the production SLM and configuration unchanged. Compare the paired raw-model report and review all eight held-out outputs; these synthetic cases are a sanity check only. Download the candidate GGUF and manifest, then follow the isolated candidate/API evaluation instructions below. Training completion alone does not authorize activation.


In [ ]:
files.download(str(quantized))
files.download(str(adapter / "training-manifest.json"))

## Next Steps
Expand and review the corpus, add diverse and multilingual fashion requests, split by garment/scenario rather than just paraphrase, evaluate on independent human-reviewed holdouts, and measure regressions. Keep the pretrained fallback label distinct from a trained adapter. Continue realistic try-on development separately under `ai-engine/`.

Sources: [Qwen model](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct), [PEFT quantization guide](https://huggingface.co/docs/peft/en/developer_guides/quantization), [llama.cpp](https://github.com/ggml-org/llama.cpp/tree/b11643).

## Validate locally before activation
This notebook has not been executed on a real GPU. Select a free T4 runtime, upload the source-only ZIP and run every cell. Training uses the pinned original Transformers checkpoint, never the Q4 GGUF.

After downloading the custom GGUF and its training manifest:
1. Keep the original runtime on port 8081. Start the candidate: `python slm-engine/tools/runtime.py serve --variant candidate --model-path C:/path/custom.gguf` (8082).
2. Restart a clean isolated evaluation harness for this run (stable fixture IDs): `npx tsx apps/api/scripts/styling-benchmark-server.ts`.
3. Run `python -m virtualfit_slm.evaluate_suite --model custom --split heldout --output docs/slm-v2-heldout-custom.json`.
4. Register only after gates pass: `python -m virtualfit_slm.register_custom --gguf C:/path/custom.gguf --manifest C:/path/training-manifest.json --baseline docs/slm-v2-heldout-base.json --candidate docs/slm-v2-heldout-custom.json`.
5. The custom selector stays unavailable until registration and runtime loading succeed. Select Original Qwen to roll back; its GGUF is never overwritten.

Compare pipeline metrics and raw-model acceptance/fallback rates. Require independent human review before making broad quality claims.